# CPU latency and threshold check
The AASIST ONNX is already only about 1.6 MB, so quantization is not assumed to be beneficial. Measure the shipped FP32 graph on labeled clips first. Select a threshold only on a calibration split, then report on a disjoint holdout. Do not use synthetic fixture performance as a general accuracy claim.

In [ ]:
!pip -q install onnxruntime==1.23.2 soundfile numpy
from pathlib import Path
import csv, json, time
import numpy as np, onnxruntime as ort, soundfile as sf
root = Path('../..')
meta = json.loads((root / 'packages/audio-detector/assets/meta.json').read_text())
session = ort.InferenceSession(str(root / 'packages/audio-detector/assets/model.onnx'), providers=['CPUExecutionProvider'])
labels = list(csv.DictReader((root / 'eval/clips/labels/role1.csv').open(encoding='utf-8', newline='')))
rows = []
for item in labels:
    audio, rate = sf.read(root / item['file'], dtype='float32', always_2d=False)
    assert rate == 16000 and audio.ndim == 1 and len(audio) >= 64600
    started = time.perf_counter(); logits = session.run(['logits'], {'wav': audio[:64600][None, :]})[0][0]
    elapsed = (time.perf_counter() - started) * 1000
    exp = np.exp(logits - np.max(logits)); fake = exp[0] / exp.sum()
    rows.append((item['label'].lower(), float(fake), elapsed))
print('clips:', len(rows), 'median inference ms:', np.median([r[2] for r in rows]) if rows else None)
print('real/fake counts:', {label: sum(r[0] == label for r in rows) for label in ('real','fake')})